# 낙서 맞히기 AI 만들기 🎨

**한국외국어대학교 인공지능 · 보너스 실습**

구글의 Quick, Draw! 게임에서는 "고양이를 그려 보세요" 같은 제시어를 받고 20초 안에 그림을 그린다. 전 세계 사람들이 그린 낙서 5천만 장 이상이 공개되어 있다. 이 노트북에서는 그 낙서로 다음을 해 본다.

| 단계 | 할 일 | 시간(T4 GPU) |
|---|---|---|
| 1 | 낙서 데이터 받아 구경하기, AI보다 먼저 맞혀 보기 | 1분 |
| 2 | 3×3 필터로 낙서 변형해 보기 | 1분 |
| 3 | CNN 학습: 10종류 낙서 분류기 | 1~2분 |
| 4 | CNN이 헷갈린 낙서, CNN 속 특성맵 보기 | 1분 |
| 5 | 삐뚤게 그려도 맞힐까? 데이터 증강 실험 | 2~3분 |
| 6 | **게임**: 내가 그린 그림을 AI가 맞히기 | 자유 |
| 7 | 보너스: 확산 모델이 낙서를 그리게 하기 | 4~6분 |

**시작 전에**: 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택한 뒤 `런타임 → 모두 실행`을 누른다. GPU 없이도 돌아가지만 3·5·7단계가 느려진다.

데이터: [Quick, Draw! Dataset](https://github.com/googlecreativelab/quickdraw-dataset) (Google, CC BY 4.0). 그림 제목과 축 이름은 Colab에서 한글 글꼴이 깨지지 않도록 영어로 표시한다.

## 0. 준비

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("gradio") is None:          # 6단계 게임에 필요
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gradio"], check=False)

import io, math, time, random, urllib.request, urllib.parse
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(GPU 없음: 느리지만 동작한다)")

## 1. 낙서 데이터 받기

카테고리 10개를 고른다. 각 카테고리 파일에는 28×28 흑백 낙서가 10만 장 넘게 들어 있는데, 앞의 6,000장만 받는다(카테고리당 약 4.7MB).

> 💡 `CATEGORIES`를 바꾸면 다른 낙서로 해 볼 수 있다. 전체 345개 목록: [categories.txt](https://github.com/googlecreativelab/quickdraw-dataset/blob/master/categories.txt)

In [ ]:
CATEGORIES = ["apple", "bicycle", "cat", "donut", "fish", "house", "smiley face", "star", "tree", "umbrella"]
KOREAN = {"apple": "사과", "bicycle": "자전거", "cat": "고양이", "donut": "도넛", "fish": "물고기",
          "house": "집", "smiley face": "웃는 얼굴", "star": "별", "tree": "나무", "umbrella": "우산"}
N_PER_CLASS = 6000            # 학습 5,000 + 평가 1,000
BASE = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/"

def fetch_doodles(name, n):
    # .npy 파일의 머리말을 읽고, 필요한 앞부분 n장만 HTTP Range 요청으로 받는다.
    url = BASE + urllib.parse.quote(name) + ".npy"
    head = urllib.request.urlopen(urllib.request.Request(url, headers={"Range": "bytes=0-1023"})).read()
    f = io.BytesIO(head)
    major, minor = np.lib.format.read_magic(f)
    reader = np.lib.format.read_array_header_1_0 if major == 1 else np.lib.format.read_array_header_2_0
    shape, _, dtype = reader(f)
    start, n = f.tell(), min(n, shape[0])
    end = start + n * 784 - 1
    body = urllib.request.urlopen(urllib.request.Request(url, headers={"Range": f"bytes={start}-{end}"})).read()
    return np.frombuffer(body, dtype=np.uint8).reshape(n, 28, 28)

t0 = time.time()
X_list, y_list = [], []
for k, name in enumerate(CATEGORIES):
    d = fetch_doodles(name, N_PER_CLASS)
    X_list.append(d); y_list.append(np.full(len(d), k))
    print(f"{name:12s} {KOREAN[name]:6s} {len(d):,}장")
X_all = np.concatenate(X_list); y_all = np.concatenate(y_list)
print(f"\n전체 {X_all.shape} · {time.time()-t0:.1f}초")

낙서를 구경해 보자. 각 줄이 한 카테고리다. 0은 검은 배경, 255는 선이다.

In [ ]:
fig, axes = plt.subplots(len(CATEGORIES), 10, figsize=(10, 10.5))
for r in range(len(CATEGORIES)):
    idx = np.random.choice(np.where(y_all == r)[0], 10, replace=False)
    for c, i in enumerate(idx):
        axes[r, c].imshow(X_all[i], cmap="gray"); axes[r, c].axis("off")
    axes[r, 0].set_title(CATEGORIES[r], fontsize=9, loc="left")
plt.tight_layout(); plt.show()

### 🕵️ 미니 게임: AI보다 먼저 맞혀 보기

아래 12장의 정답을 종이에 적어 보자. 다음 셀을 실행하면 정답이 나온다. 몇 개 맞혔는지 기억해 두고, 3단계에서 CNN의 점수와 비교한다.

In [ ]:
quiz_idx = np.random.RandomState(2026).choice(len(X_all), 12, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(10, 3.8))
for ax, i, q in zip(axes.flat, quiz_idx, range(1, 13)):
    ax.imshow(X_all[i], cmap="gray"); ax.set_title(f"Q{q}"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
for q, i in enumerate(quiz_idx, 1):
    print(f"Q{q:2d}: {CATEGORIES[y_all[i]]} ({KOREAN[CATEGORIES[y_all[i]]]})")

### 카테고리별 평균 낙서

같은 카테고리 6,000장을 픽셀마다 평균 내면 사람들이 "보통" 어떻게 그리는지 보인다. 밝을수록 많은 사람이 그 자리에 선을 그었다.

In [ ]:
fig, axes = plt.subplots(1, len(CATEGORIES), figsize=(12, 1.8))
for k, ax in enumerate(axes):
    ax.imshow(X_all[y_all == k].mean(0), cmap="magma"); ax.set_title(CATEGORIES[k], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. 필터 놀이: 3×3 커널로 낙서 바꾸기

CNN의 합성곱은 3×3 커널을 이미지 위로 밀며 곱하고 더하는 연산이다. 사람이 정한 커널을 몇 개 써 보자. `MY_KERNEL`의 숫자를 바꿔 나만의 필터도 만들어 본다.

In [ ]:
KERNELS = {
    "edge (Laplacian)": [[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]],
    "vertical (Sobel x)": [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
    "horizontal (Sobel y)": [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
    "blur": [[1/9] * 3] * 3,
    "MY_KERNEL": [[0, 1, 0], [1, -4, 1], [0, 1, 0]],   # ← 숫자를 바꿔 보자
}
samples = [np.where(y_all == CATEGORIES.index(n))[0][3] for n in ["cat", "house", "bicycle", "star"]]
x = torch.tensor(X_all[samples], dtype=torch.float32).unsqueeze(1) / 255.0

fig, axes = plt.subplots(len(samples), len(KERNELS) + 1, figsize=(12, 8))
for r in range(len(samples)):
    axes[r, 0].imshow(x[r, 0], cmap="gray"); axes[r, 0].set_title("original" if r == 0 else ""); axes[r, 0].axis("off")
for c, (name, k) in enumerate(KERNELS.items(), 1):
    w = torch.tensor(k, dtype=torch.float32).view(1, 1, 3, 3)
    out = F.conv2d(x, w, padding=1)
    for r in range(len(samples)):
        axes[r, c].imshow(out[r, 0].abs() if "blur" not in name else out[r, 0], cmap="gray")
        axes[r, c].set_title(name if r == 0 else "", fontsize=9); axes[r, c].axis("off")
plt.tight_layout(); plt.show()

## 3. CNN 학습: 낙서 10종류 분류기

카테고리마다 5,000장으로 학습하고 1,000장으로 평가한다. 구조는 합성곱 블록 2개와 완전연결층 2개다.

```
입력 1×28×28 → Conv 3×3 (32) → ReLU → MaxPool → 32×14×14
            → Conv 3×3 (64) → ReLU → MaxPool → 64×7×7 → 펼치기 3136 → 128 → 10
```

In [ ]:
perm = np.random.RandomState(0).permutation(len(X_all))
X_all, y_all = X_all[perm], y_all[perm]
is_test = np.zeros(len(X_all), bool)
for k in range(len(CATEGORIES)):
    is_test[np.where(y_all == k)[0][:1000]] = True
is_test |= np.isin(perm, quiz_idx)          # 미니 게임 12장은 학습에 쓰지 않는다
Xtr = torch.tensor(X_all[~is_test], dtype=torch.float32).unsqueeze(1).div(255).to(device)
ytr = torch.tensor(y_all[~is_test]).long().to(device)
Xte = torch.tensor(X_all[is_test], dtype=torch.float32).unsqueeze(1).div(255).to(device)
yte = torch.tensor(y_all[is_test]).long().to(device)
print("학습", tuple(Xtr.shape), "평가", tuple(Xte.shape))

def make_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, len(CATEGORIES)),
    ).to(device)

@torch.no_grad()
def accuracy(model, X, y, bs=1000):
    model.eval()
    hit = sum((model(X[i:i+bs]).argmax(1) == y[i:i+bs]).sum().item() for i in range(0, len(X), bs))
    return hit / len(X)

def train(model, epochs=4, augment=None, bs=128, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); order = torch.randperm(len(Xtr), device=device); tot = 0
        for i in range(0, len(Xtr), bs):
            b = order[i:i+bs]; xb = Xtr[b]
            if augment is not None: xb = augment(xb)
            loss = F.cross_entropy(model(xb), ytr[b])
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(b)
        print(f"epoch {ep}: 손실 {tot/len(Xtr):.3f} · 평가 정확도 {accuracy(model, Xte, yte):.1%} · {time.time()-t0:.1f}초")
    return model

torch.manual_seed(SEED)
cnn = make_cnn()
print("파라미터 수:", f"{sum(p.numel() for p in cnn.parameters()):,}")
cnn = train(cnn)

미니 게임에서 내가 맞힌 개수와 비교해 보자. 같은 12장을 CNN은 몇 개 맞힐까?

In [ ]:
@torch.no_grad()
def predict(model, imgs_uint8):
    model.eval()
    x = torch.tensor(np.asarray(imgs_uint8), dtype=torch.float32).view(-1, 1, 28, 28).div(255).to(device)
    return F.softmax(model(x), 1).cpu().numpy()

quiz_imgs = np.concatenate(X_list)[quiz_idx]; quiz_true = np.concatenate(y_list)[quiz_idx]
p = predict(cnn, quiz_imgs)
for q, (t, pr) in enumerate(zip(quiz_true, p), 1):
    k = pr.argmax(); mark = "O" if k == t else "X"
    print(f"Q{q:2d} {mark}  정답 {KOREAN[CATEGORIES[t]]:5s}  CNN {KOREAN[CATEGORIES[k]]:5s} ({pr[k]:.0%})")
print(f"\nCNN 점수: {(p.argmax(1) == quiz_true).sum()} / 12")

## 4. CNN은 무엇을 헷갈렸을까?

혼동행렬의 (행, 열) 칸은 "정답이 행인데 열로 예측한 개수"다. 대각선 밖에서 숫자가 큰 칸이 자주 헷갈린 쌍이다.

In [ ]:
with torch.no_grad():
    cnn.eval(); pred = torch.cat([cnn(Xte[i:i+1000]).argmax(1) for i in range(0, len(Xte), 1000)]).cpu().numpy()
true = yte.cpu().numpy(); K = len(CATEGORIES)
cm = np.zeros((K, K), int)
for t, q in zip(true, pred): cm[t, q] += 1

plt.figure(figsize=(7, 6)); plt.imshow(cm, cmap="Blues")
plt.xticks(range(K), CATEGORIES, rotation=60); plt.yticks(range(K), CATEGORIES)
for i in range(K):
    for j in range(K):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="white" if cm[i, j] > 500 else "black")
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Confusion matrix (test 10,000)"); plt.tight_layout(); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
print("가장 많이 헷갈린 쌍 TOP 5")
for f in np.argsort(off.ravel())[::-1][:5]:
    i, j = divmod(f, K); print(f"  {KOREAN[CATEGORIES[i]]} → {KOREAN[CATEGORIES[j]]}: {off[i, j]}장")

틀린 낙서를 직접 보자. 사람이 봐도 헷갈리는 것이 있고, 사람은 쉽게 맞히는데 CNN이 틀린 것도 있다.

In [ ]:
wrong = np.where(pred != true)[0][:15]
Xte_np = (Xte.cpu().numpy()[:, 0] * 255).astype(np.uint8)
fig, axes = plt.subplots(3, 5, figsize=(10, 6.6))
for ax, i in zip(axes.flat, wrong):
    ax.imshow(Xte_np[i], cmap="gray"); ax.axis("off")
    ax.set_title(f"true: {CATEGORIES[true[i]]}\npred: {CATEGORIES[pred[i]]}", fontsize=8.5)
plt.tight_layout(); plt.show()

### CNN 속 들여다보기

첫 번째 합성곱층이 **학습으로 스스로 만든** 3×3 필터 32개와, 낙서 한 장이 그 필터를 지난 특성맵 32장이다. 2단계에서 사람이 만든 필터와 비교해 보자.

In [ ]:
W1 = cnn[0].weight.detach().cpu()[:, 0]
fig, axes = plt.subplots(2, 16, figsize=(12, 1.8))
for ax, w in zip(axes.flat, W1): ax.imshow(w, cmap="bwr", vmin=-W1.abs().max(), vmax=W1.abs().max()); ax.axis("off")
plt.suptitle("Learned 3x3 filters (layer 1, red=+, blue=-)", fontsize=10); plt.show()

i = int(np.where(true == CATEGORIES.index("bicycle"))[0][0])
with torch.no_grad(): fmap = F.relu(cnn[0](Xte[i:i+1]))[0].cpu()
fig, axes = plt.subplots(3, 11, figsize=(12, 3.6))
axes[0, 0].imshow(Xte_np[i], cmap="gray"); axes[0, 0].set_title("input", fontsize=8)
for ax, f in zip(list(axes.flat)[1:], fmap): ax.imshow(f, cmap="viridis")
for ax in axes.flat: ax.axis("off")
plt.suptitle("Feature maps of a bicycle (32 maps, 28x28)", fontsize=10); plt.tight_layout(); plt.show()

## 5. 삐뚤게 그려도 맞힐까? 데이터 증강 실험

평가 낙서를 ±35도 회전시키고 위치를 조금 옮기면 정확도가 어떻게 될까? 그리고 학습할 때 무작위로 돌리고 옮긴 낙서를 보여 주면(데이터 증강) 나아질까?

In [ ]:
def random_affine(x, max_deg=35, max_shift=0.12, gen=None):
    # 한 배치 안의 낙서마다 다른 각도·위치로 회전·이동한다 (GPU에서 한 번에 계산).
    n = x.shape[0]
    ang = (torch.rand(n, device=x.device, generator=gen) * 2 - 1) * math.radians(max_deg)
    sh = (torch.rand(n, 2, device=x.device, generator=gen) * 2 - 1) * max_shift * 2
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.stack([torch.stack([cos, -sin, sh[:, 0]], 1), torch.stack([sin, cos, sh[:, 1]], 1)], 1)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, align_corners=False, padding_mode="zeros")

g = torch.Generator(device=device).manual_seed(123)
Xte_tilt = random_affine(Xte, gen=g)

fig, axes = plt.subplots(2, 8, figsize=(10, 2.8))
for c in range(8):
    axes[0, c].imshow(Xte[c, 0].cpu(), cmap="gray"); axes[1, c].imshow(Xte_tilt[c, 0].cpu(), cmap="gray")
    axes[0, c].axis("off"); axes[1, c].axis("off")
axes[0, 0].set_title("original", loc="left", fontsize=9); axes[1, 0].set_title("tilted + shifted", loc="left", fontsize=9)
plt.tight_layout(); plt.show()

print(f"기본 CNN · 원래 평가 낙서: {accuracy(cnn, Xte, yte):.1%}")
print(f"기본 CNN · 삐뚤어진 평가 낙서: {accuracy(cnn, Xte_tilt, yte):.1%}")

In [ ]:
torch.manual_seed(SEED)
cnn_aug = train(make_cnn(), epochs=6, augment=random_affine)
print()
print(f"{'':16s}{'원래 낙서':>10s}{'삐뚤어진 낙서':>14s}")
for name, m in [("기본 CNN", cnn), ("증강 CNN", cnn_aug)]:
    print(f"{name:16s}{accuracy(m, Xte, yte):>10.1%}{accuracy(m, Xte_tilt, yte):>14.1%}")

> 🤔 증강 CNN은 삐뚤어진 낙서에서 얼마나 나아졌는가? 원래 낙서의 정확도는 어떻게 되었는가? 회전 각도 `max_deg`를 90이나 180으로 키우면 어떤 카테고리가 손해를 볼까? (힌트: 우산과 나무를 뒤집으면?)

## 6. 🎮 게임: 내가 그린 낙서를 AI가 맞힐까?

아래 셀을 실행하면 그림판이 나온다. 10개 카테고리 중 하나를 마우스(또는 손가락)로 그리고 결과를 확인한다. 그림판 가운데에 크게 그리면 잘 맞힌다.

- 일부러 AI를 속여 보자: 고양이처럼 보이는 집을 그릴 수 있을까?
- `MODEL = cnn_aug`로 바꾸면 증강 CNN으로 게임한다.

In [ ]:
MODEL = cnn

def to_doodle(img):
    # 그림판 이미지 → Quick, Draw! 형식(검은 배경, 흰 선, 28×28, 그림을 가운데에 맞춤)
    if isinstance(img, dict):
        img = img.get("composite", img.get("image"))
    a = np.asarray(img).astype(np.float32)
    if a.ndim == 3:
        if a.shape[2] == 4:                                   # 투명 배경 → 알파 채널이 곧 선
            ink = a[..., 3] * (255 - a[..., :3].mean(2)) / 255
        else:
            ink = 255 - a[..., :3].mean(2)
    else:
        ink = 255 - a if a.mean() > 127 else a
    ys, xs = np.where(ink > 30)
    if len(ys) == 0: return None
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    s = max(y1 - y0, x1 - x0) + 1; pad = int(s * 0.12)
    canvas = np.zeros((s + 2 * pad, s + 2 * pad), np.float32)
    oy, ox = pad + (s - (y1 - y0 + 1)) // 2, pad + (s - (x1 - x0 + 1)) // 2
    canvas[oy:oy + y1 - y0 + 1, ox:ox + x1 - x0 + 1] = ink[y0:y1 + 1, x0:x1 + 1]
    t = torch.tensor(canvas)[None, None]
    small = F.interpolate(t, size=(28, 28), mode="area")[0, 0].numpy()
    return np.clip(small / max(small.max(), 1e-6) * 255, 0, 255).astype(np.uint8)

def guess(img):
    d = to_doodle(img)
    if d is None: return {"그림을 그려 주세요": 1.0}
    p = predict(MODEL, d[None])[0]
    return {f"{KOREAN[c]} ({c})": float(p[k]) for k, c in enumerate(CATEGORIES)}

import gradio as gr
try:
    pad = gr.Sketchpad(canvas_size=(320, 320), type="numpy",
                       brush=gr.Brush(default_size=14, colors=["#000000"], color_mode="fixed"))
except Exception:
    pad = gr.Sketchpad(type="numpy")
demo = gr.Interface(fn=guess, inputs=pad, outputs=gr.Label(num_top_classes=3), live=True,
                    title="낙서 맞히기 AI", description=" · ".join(KOREAN[c] for c in CATEGORIES) + " 중 하나를 그려 보세요")
demo.launch(debug=False)

## 7. 보너스: 확산 모델이 낙서를 그리게 하기

지금까지는 낙서를 **알아보는** AI였다. 이번에는 낙서를 **그리는** AI를 만든다. 확산 모델은 낙서에 잡음을 조금씩 섞어 완전한 잡음으로 만드는 과정을 배운 뒤, 거꾸로 잡음에서 잡음을 조금씩 걷어 내며 새 낙서를 만든다.

먼저 잡음을 섞는 과정(정방향)을 눈으로 확인한다. `GEN_CLASS`를 바꾸면 다른 낙서를 학습한다.

In [ ]:
GEN_CLASS = "smiley face"        # 다른 카테고리로 바꿔 보자: "star", "fish", "house" …
T = 400
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)

data = torch.tensor(np.concatenate(X_list)[np.concatenate(y_list) == CATEGORIES.index(GEN_CLASS)],
                    dtype=torch.float32, device=device).unsqueeze(1) / 127.5 - 1      # [-1, 1]
data = F.pad(data, (2, 2, 2, 2), value=-1)                                              # 32×32
print("학습 낙서:", tuple(data.shape))

x0 = data[:1]; steps = [0, 25, 50, 100, 200, 300, 399]
fig, axes = plt.subplots(1, len(steps), figsize=(11, 1.9))
for ax, t in zip(axes, steps):
    xt = abar[t].sqrt() * x0 + (1 - abar[t]).sqrt() * torch.randn_like(x0)
    ax.imshow(xt[0, 0].cpu(), cmap="gray", vmin=-1, vmax=1); ax.set_title(f"t={t}", fontsize=9); ax.axis("off")
plt.suptitle("Forward process: x_t = sqrt(abar_t) x_0 + sqrt(1-abar_t) eps", fontsize=10); plt.tight_layout(); plt.show()

잡음 예측 신경망은 작은 U-Net 형태다. 잡음이 섞인 낙서 x<sub>t</sub>와 단계 t를 받아 섞인 잡음 ε을 맞히도록 학습한다(손실: 평균제곱오차). T4 GPU에서 약 3~5분 걸린다.

In [ ]:
class TimeEmb(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 2), nn.SiLU(), nn.Linear(dim * 2, dim * 2))
    def forward(self, t):
        half = self.dim // 2
        f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        e = t.float()[:, None] * f[None]
        return self.mlp(torch.cat([e.sin(), e.cos()], 1))

class Block(nn.Module):
    def __init__(self, cin, cout, tdim):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, padding=1); self.c2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.n1 = nn.GroupNorm(8, cout); self.n2 = nn.GroupNorm(8, cout); self.t = nn.Linear(tdim, cout)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x, te):
        h = F.silu(self.n1(self.c1(x))) + self.t(te)[:, :, None, None]
        return F.silu(self.n2(self.c2(h))) + self.skip(x)

class TinyUNet(nn.Module):
    def __init__(self, c=64, tdim=128):
        super().__init__()
        self.te = TimeEmb(tdim // 2)
        self.inc = nn.Conv2d(1, c, 3, padding=1)
        self.d1 = Block(c, c, tdim); self.d2 = Block(c, 2 * c, tdim); self.mid = Block(2 * c, 2 * c, tdim)
        self.u2 = Block(4 * c, c, tdim); self.u1 = Block(2 * c, c, tdim)
        self.out = nn.Conv2d(c, 1, 3, padding=1)
    def forward(self, x, t):
        te = self.te(t)
        h0 = self.d1(self.inc(x), te)                       # 32×32
        h1 = self.d2(F.avg_pool2d(h0, 2), te)               # 16×16
        m = self.mid(F.avg_pool2d(h1, 2), te)               # 8×8
        u = self.u2(torch.cat([F.interpolate(m, scale_factor=2), h1], 1), te)
        u = self.u1(torch.cat([F.interpolate(u, scale_factor=2), h0], 1), te)
        return self.out(u)

torch.manual_seed(SEED)
eps_model = TinyUNet().to(device)
opt = torch.optim.AdamW(eps_model.parameters(), lr=2e-3)
STEPS = 3000 if device == "cuda" else 600
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=STEPS)
print("파라미터 수:", f"{sum(p.numel() for p in eps_model.parameters()):,}", "· 학습 스텝:", STEPS)

t0 = time.time(); run = 0
for step in range(1, STEPS + 1):
    x0 = data[torch.randint(0, len(data), (128,), device=device)]
    t = torch.randint(0, T, (len(x0),), device=device)
    eps = torch.randn_like(x0)
    xt = abar[t].sqrt()[:, None, None, None] * x0 + (1 - abar[t]).sqrt()[:, None, None, None] * eps
    loss = F.mse_loss(eps_model(xt, t), eps)
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    run = 0.98 * run + 0.02 * loss.item() if step > 1 else loss.item()
    if step % 500 == 0 or step == STEPS:
        print(f"step {step:5d} · 잡음 예측 손실 {run:.4f} · {time.time()-t0:.0f}초")

이제 순수한 잡음에서 출발해 400단계에 걸쳐 잡음을 걷어 낸다(DDPM 샘플링). 중간 과정도 함께 본다.

In [ ]:
@torch.no_grad()
def sample(n=16, seed=0, keep=(399, 300, 200, 100, 50, 0)):
    g = torch.Generator(device=device).manual_seed(seed)
    x = torch.randn(n, 1, 32, 32, device=device, generator=g); snaps = {}
    eps_model.eval()
    for t in range(T - 1, -1, -1):
        tt = torch.full((n,), t, device=device, dtype=torch.long)
        e = eps_model(x, tt)
        x = (x - betas[t] / (1 - abar[t]).sqrt() * e) / alphas[t].sqrt()
        if t > 0: x = x + betas[t].sqrt() * torch.randn(x.shape, device=device, generator=g)
        if t in keep: snaps[t] = x.clamp(-1, 1).cpu()
    return x.clamp(-1, 1).cpu(), snaps

gen, snaps = sample(16, seed=1)
fig, axes = plt.subplots(4, 6, figsize=(9, 6.4))
for r in range(4):
    for c, t in enumerate(sorted(snaps, reverse=True)):
        axes[r, c].imshow(snaps[t][r, 0], cmap="gray", vmin=-1, vmax=1); axes[r, c].axis("off")
        if r == 0: axes[r, c].set_title(f"t={t}", fontsize=9)
plt.suptitle("Reverse process: noise -> doodle", fontsize=10); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 8, figsize=(10, 2.8))
for ax, im in zip(axes.flat, gen): ax.imshow(im[0], cmap="gray", vmin=-1, vmax=1); ax.axis("off")
plt.suptitle(f"16 new '{GEN_CLASS}' doodles drawn by the diffusion model", fontsize=10); plt.tight_layout(); plt.show()

확산 모델이 그린 낙서를 3단계의 CNN에게 보여 주자. CNN이 의도한 카테고리로 알아보면 그럴듯하게 그린 것이다.

In [ ]:
gen_u8 = ((gen[:, 0, 2:30, 2:30] + 1) * 127.5).clamp(0, 255).numpy().astype(np.uint8)
p = predict(cnn, gen_u8)
hit = (p.argmax(1) == CATEGORIES.index(GEN_CLASS)).mean()
print(f"CNN이 '{GEN_CLASS}'로 알아본 비율: {hit:.0%} (16장 중 {int(round(hit * 16))}장)")
for k in np.argsort(-np.bincount(p.argmax(1), minlength=len(CATEGORIES)))[:3]:
    print(f"  {KOREAN[CATEGORIES[k]]}: {(p.argmax(1) == k).sum()}장")

## 8. 도전 과제

1. `CATEGORIES`를 내가 좋아하는 낙서 10개로 바꿔 처음부터 다시 실행한다. 정확도가 가장 낮은 조합을 찾아본다.
2. 3단계의 CNN에 합성곱 블록을 하나 더 넣거나 채널 수를 바꿔 파라미터 수와 정확도를 비교한다.
3. 5단계의 `max_deg`를 0, 15, 35, 90으로 바꿔 원래 낙서와 삐뚤어진 낙서의 정확도를 표로 정리한다.
4. 6단계 게임에서 AI가 틀리게 만드는 낙서를 3개 찾아 캡처한다. 왜 틀렸을지 4단계의 혼동행렬과 연결해 설명한다.
5. 7단계에서 `GEN_CLASS`를 바꾸거나 `STEPS`를 늘려 생성 품질이 어떻게 달라지는지 비교한다. CNN이 알아본 비율을 품질 지표로 써 본다.

*데이터: Google Creative Lab, The Quick, Draw! Dataset (CC BY 4.0).*